In [5]:
import pandas as pd
import numpy as np
import janitor

In [6]:
nd1 = pd.read_excel("raw_night_1854_1855.xlsx")
nd2 = pd.read_excel("raw_night_1855_1856.xlsx")

In [7]:
nd1.head()

,Month,Avg Army Size,Disease Deaths,Wounds,Other Causes
0,Apr 1854,8571,1,0,5
1,May 1854,23333,12,0,9
2,Jun 1854,28333,11,0,6
3,Jul 1854,28722,359,0,23
4,Aug 1854,30246,828,1,30


In [8]:
nd1.dtypes

Month             object
Avg Army Size      int64
Disease Deaths     int64
Wounds             int64
Other Causes       int64
dtype: object

In [9]:
#CLEANING THE DATASET USING THE TINY DATA CEANING RULE OF ONE VARIABLE PER ROW, WE SEPARATE THE MONTH AND YEAR COLUMN
nd1a = (nd1.Month.
    str.split(" ", expand = True).
    rename(columns = {0 : "Month_Name",
                     1 : "Year"}))


In [10]:
nd1a.head()

,Month_Name,Year
0,Apr,1854
1,May,1854
2,Jun,1854
3,Jul,1854
4,Aug,1854


In [11]:
#JOINING IT TO THE ORIGINAL DATA
nd1c = pd.concat([nd1, nd1a], axis = 1)

In [12]:
nd1c

,Month,Avg Army Size,Disease Deaths,Wounds,Other Causes,Month_Name,Year
0,Apr 1854,8571,1,0,5,Apr,1854
1,May 1854,23333,12,0,9,May,1854
2,Jun 1854,28333,11,0,6,Jun,1854
3,Jul 1854,28722,359,0,23,Jul,1854
4,Aug 1854,30246,828,1,30,Aug,1854
5,Sep 1854,30290,788,81,70,Sep,1854
6,Oct 1854,30643,503,132,128,Oct,1854
7,Nov 1854,27543,844,287,106,Nov,1854
8,Dec 1854,32393,1725,114,131,Dec,1854
9,Jan 1855,32393,2761,83,324,Jan,1855


In [183]:
#COMBINING THE TWO DATA SETS

#nd_all = pd.concat([nd1c, nd2c], ignore_index=True)
#nd_all

In [184]:
#SUMMARY STATISTICS
#nd_all.describe()

In [13]:
from lets_plot import *
LetsPlot.setup_html()

In [14]:
nd1d = nd1c.pivot_longer(index = ["Month", "Avg Army Size", "Month_Name", "Year"],
                 names_to = "Cause Of Death",
                 values_to = "Count")


nd1d.head()

,Month,Avg Army Size,Month_Name,Year,Cause Of Death,Count
0,Apr 1854,8571,Apr,1854,Disease Deaths,1
1,May 1854,23333,May,1854,Disease Deaths,12
2,Jun 1854,28333,Jun,1854,Disease Deaths,11
3,Jul 1854,28722,Jul,1854,Disease Deaths,359
4,Aug 1854,30246,Aug,1854,Disease Deaths,828


In [15]:
nd1d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Month           36 non-null     object
 1   Avg Army Size   36 non-null     int64 
 2   Month_Name      36 non-null     object
 3   Year            36 non-null     object
 4   Cause Of Death  36 non-null     object
 5   Count           36 non-null     int64 
dtypes: int64(2), object(4)
memory usage: 1.8+ KB


In [16]:
#CONVERTING THE COUNT COLUMN TO INT
nd1d["Count"] = nd1d["Count"].astype(int)
nd1d.dtypes

Month             object
Avg Army Size      int64
Month_Name        object
Year              object
Cause Of Death    object
Count              int64
dtype: object

In [318]:
nd1d.describe()

,Avg Army Size,Count,Sqrt_Count
count,36.000000,36.000000,36.000000
mean,27943.333333,369.333333,13.604657
std,6422.886071,643.328843,13.766293
min,8571.000000,0.000000,0.000000
25%,28135.500000,10.500000,3.237469
50%,30268.000000,94.500000,9.703032
75%,31080.500000,359.500000,18.960471
max,32393.000000,2761.000000,52.545219


In [17]:
nd1d["Sqrt_Count"] = np.sqrt(nd1d["Count"])
nd1d.head()

,Month,Avg Army Size,Month_Name,Year,Cause Of Death,Count,Sqrt_Count
0,Apr 1854,8571,Apr,1854,Disease Deaths,1,1.000000
1,May 1854,23333,May,1854,Disease Deaths,12,3.464102
2,Jun 1854,28333,Jun,1854,Disease Deaths,11,3.316625
3,Jul 1854,28722,Jul,1854,Disease Deaths,359,18.947295
4,Aug 1854,30246,Aug,1854,Disease Deaths,828,28.774989


In [65]:
# Rename short month names to full capitalised names
month_name_map = {
    "Apr": "APRIL", "May": "MAY", "Jun": "JUNE",
    "Jul": "JULY", "Aug": "AUGUST", "Sep": "SEPTEMBER",
    "Oct": "OCTOBER", "Nov": "NOVEMBER", "Dec": "DECEMBER",
    "Jan": "JANUARY", "Feb": "FEBRUARY", "Mar": "MARCH"
}

nd1d["Month_Name"] = nd1d["Month_Name"].map(month_name_map)

In [116]:
# Using pd.Categorical to arrange the month name in the correct order

#month_order = ["Jul","Aug","Sep","Oct","Nov","Dec","Jan","Feb","Mar","Apr", "May","Jun"]

#nd1d["Month_Name"] = pd.Categorical(nd1d["Month_Name"], 
#                            categories=month_order, ordered=True)

month_order = [
   "JULY", "AUGUST", "SEPTEMBER",
    "OCTOBER", "NOVEMBER", "DECEMBER", "JANUARY", "FEBRUARY", "MARCH",  "APRIL", "MAY", "JUNE"
]

nd1d["Month_Name"] = pd.Categorical(
    nd1d["Month_Name"],
    categories=month_order,
    ordered=True
)

nd1d["Cause Of Death"] = pd.Categorical(nd1d["Cause Of Death"],
    categories=["Wounds", "Other Causes","Disease Deaths"],
    ordered=True
)

In [119]:
p1 = (ggplot(nd1d, aes("Month_Name", "Sqrt_Count", fill="Cause Of Death"))
+ geom_bar(stat="identity", alpha=0.6)
+ scale_fill_manual(values=["#E5BDB0", "#7C716D", "#8AABB9"])
#+ coord_polar()
+ coord_polar(ylim=[0, 28])

+ labs(
    fill="Cause Of Death",
    subtitle="April 1854 to March 1855",
     caption ='Johanna Ezedinma')

+ theme(
    plot_subtitle=element_text(hjust=0.5),
    axis_text_y=element_blank(),
    axis_title_y=element_blank(),
    axis_title_x=element_blank(),
    axis_ticks=element_blank(),
    panel_grid=element_blank(),
    panel_border=element_blank(),
    axis_line=element_blank(),
    plot_caption=element_text(color="lightgrey"),
    legend_position="bottom",
    axis_text_x=element_text(size=9, margin=[-5, 0, -5, 0]),
    plot_margin=[0, 0, 0, 0]
    
  )
+ ggsize(600, 900)
)
p1

In [61]:
nd2.head()

,Month,Avg Army Size,Disease Deaths,Wounds,Other Causes
0,Apr 1855,32393,477,48,57
1,May 1855,35693,508,32,37
2,Jun 1855,40253,802,31,31
3,Jul 1855,35786,382,30,33
4,Aug 1855,51074,483,41,55


In [25]:
nd2.dtypes

Month             object
Avg Army Size      int64
Disease Deaths     int64
Wounds             int64
Other Causes       int64
dtype: object

In [57]:
#CLEANING THE DATASET USING THE TINY DATA CEANING RULE OF ONE VARIABLE PER ROW, WE SEPARATE THE MONTH AND YEAR COLUMN
nd2a = (nd1.Month.
    str.split(" ", expand = True).
    rename(columns = {0 : "Month_Name",
                     1 : "Year"}))


In [118]:
nd2a.head()

,Month_Name,Year
0,Apr,1854
1,May,1854
2,Jun,1854
3,Jul,1854
4,Aug,1854


In [29]:
#JOINING IT TO THE ORIGINAL DATA
nd2c = pd.concat([nd2, nd2a], axis = 1)
nd2c

,Month,Avg Army Size,Disease Deaths,Wounds,Other Causes,Month_Name,Year
0,Apr 1855,32393,477,48,57,Apr,1854
1,May 1855,35693,508,32,37,May,1854
2,Jun 1855,40253,802,31,31,Jun,1854
3,Jul 1855,35786,382,30,33,Jul,1854
4,Aug 1855,51074,483,41,55,Aug,1854
5,Sep 1855,52280,189,47,37,Sep,1854
6,Oct 1855,53321,128,36,33,Oct,1854
7,Nov 1855,45679,178,19,18,Nov,1854
8,Dec 1855,36241,91,15,15,Dec,1854
9,Jan 1856,34292,42,2,3,Jan,1855


In [35]:
nd2d = nd2c.pivot_longer(index = ["Month", "Avg Army Size", "Month_Name", "Year"],
                 names_to = "Cause Of Death",
                 values_to = "Count")


nd2d.head()

,Month,Avg Army Size,Month_Name,Year,Cause Of Death,Count
0,Apr 1855,32393,Apr,1854,Disease Deaths,477
1,May 1855,35693,May,1854,Disease Deaths,508
2,Jun 1855,40253,Jun,1854,Disease Deaths,802
3,Jul 1855,35786,Jul,1854,Disease Deaths,382
4,Aug 1855,51074,Aug,1854,Disease Deaths,483


In [36]:
#CONVERTING THE COUNT COLUMN TO INT
nd2d["Count"] = nd2d["Count"].astype(int)
nd2d.dtypes

Month             object
Avg Army Size      int64
Month_Name        object
Year              object
Cause Of Death    object
Count              int64
dtype: object

In [37]:
nd2d["Sqrt_Count"] = np.sqrt(nd2d["Count"])
nd2d.head()

,Month,Avg Army Size,Month_Name,Year,Cause Of Death,Count,Sqrt_Count
0,Apr 1855,32393,Apr,1854,Disease Deaths,477,21.840330
1,May 1855,35693,May,1854,Disease Deaths,508,22.538855
2,Jun 1855,40253,Jun,1854,Disease Deaths,802,28.319605
3,Jul 1855,35786,Jul,1854,Disease Deaths,382,19.544820
4,Aug 1855,51074,Aug,1854,Disease Deaths,483,21.977261


In [87]:
# Using pd.Categorical to arrange the month name 

month_order = [
    "JULY", "AUGUST", "SEPTEMBER",
    "OCTOBER", "NOVEMBER", "DECEMBER", "JANUARY", "FEBRUARY", "MARCH", "APRIL", "MAY", "JUNE"
]

nd2d["Month_Name"] = pd.Categorical(nd1d["Month_Name"], 
                            categories=month_order, ordered=True)

nd2d["Cause Of Death"] = pd.Categorical(nd2d["Cause Of Death"],
    categories=["Wounds", "Other Causes","Disease Deaths"],
    ordered=True
)

In [113]:
p2 = (ggplot(nd2d, aes("Month_Name", "Sqrt_Count", fill="Cause Of Death"))
+ geom_bar(stat="identity", alpha=0.6)
+ scale_fill_manual(values=["#E5BDB0", "#7C716D","#8AABB9"])  
+ coord_polar(ylim=[0, 28])
+ labs(
    fill="Cause Of Death",
    subtitle = "April 1855 to March 1856",)
#+ guides(fill="none")
+ theme(
    plot_subtitle=element_text(hjust=0.5),
    axis_text_y=element_blank(),
    axis_title_y=element_blank(),
    axis_title_x=element_blank(),
    axis_ticks=element_blank(),
    panel_grid=element_blank(),
    panel_border=element_blank(),
    axis_line = element_blank(),
    legend_position="bottom",
    axis_text_x=element_text(size=9, margin=[-5, 0, -5, 0]),
    plot_margin=[0, 0, 0, 0]
  )
+ guides(fill="none")   #to remove the legend
+ ggsize(600, 900)
)
p2


In [122]:
final_grid = (gggrid([p2, p1], ncol=2, hspace=0.5)
+ ggtitle("Diagram of the Causes of Mortality in the Army in the East Between 1854 to 1856")
+ theme(
    plot_title=element_text(
       face="bold", 
       hjust=0.5,
     margin=[20, 0, 24, 0]
  )

 )
+ ggsize(1500, 700)
)
final_grid

In [123]:
ggsave(final_grid, 'my_final_chart.png')

'C:\\Users\\ASUS\\Desktop\\Nightingale Mini Project\\lets-plot-images\\my_final_chart.png'

# Reflection

Most of the deaths during the Crimean War were caused by preventable diseases. 
The early months of April to June 1854 shows very few deaths by preventable disease reflecting the fact that British troops had only just arrived and the devastating outbreak had not yet begun. 

However, from July 1854 onwards, preventable disease deaths rose sharply and dramatically. The periods of December 1854, January 1855, and February 1855 recorded the highest disease related deaths, with January 1855 alone accounting for 2,761 deaths. 
Deaths caused by wounds and other causes were much smaller in comparison, clearly showing that soldiers were dying far more from preventable diseases than from actual battle injuries.

Between 1855 and 1856, there was a significant decrease in deaths from preventable disease compared to the previous year. 
This reduction was not instant as seen in the diagram, it happened gradually through 1855 as Florence Nightingale's sanitation reforms began to take effect. The early months of the second period still show elevated disease deaths, but by late 1855 and early 1856 the blue wedges almost disappear entirely. 
This gradual shrinking demonstrates that the reforms were working month by month, and that most of the earlier deaths had been entirely avoidable.

This visualization also highlights why a chart is more powerful than a plain table of numbers. A government official glancing at rows of data may not grasp the scale of the problem but one look at the two coxcomb charts tells the entire story instantly.